# EXP-2026-001 on Colab: Stage 5 pilot, part 2 of 2 (validation comparison)

This notebook loads the step-1000 UNet from part 1 and generates every axial slice of each accepted validation patient. The plan was fixed in `configs/experiments/EXP-2026-001.json` (block `stage5_pilot`) before any Stage 5 run.

| Sampler | Starts from | Seeds | Role |
|---|---|---|---|
| `pair_preop` | the noised pre-op image and mask at t = 981 | 11, 22, 33 | primary |
| `pure_noise` | pure noise in both streams | 11 | the original MSF start |

Each result is mapped back to the 1 mm frame and scored against the real post-op scan with the frozen rules: the MRI copy control, the primary ROI within 10 mm of the pre-op labels, the stable ROI beyond 30 mm, and a patient-paired bootstrap. An intensity-matched copy is reported next to it: the pre-op image after the best global intensity map to the real target. It is an oracle control that shows how much a global intensity change alone could gain, and the model must beat it for a replication to be justified. Generation reads only pre-op data plus the class text. That text comes from the post-op label, so it is an **oracle condition**. Post-op images are only scored against.

The reading can justify a replication with train seeds 2027 and 2028, or say ITERATE. With at most three validation patients it cannot PROMOTE, and it supports no clinical claim. The test set stays sealed.

Expect about 2 hours. Outputs are written to Colab's local disk first and copied to `synthesis/EXP-2026-001/stage5/<RUN_TAG>/` on Drive (about 2 GB) after every stack, so a Drive disconnect does not stop the run. After a lost session, run the notebook again with the same `RUN_TAG` and it continues from the stacks on Drive. Keep the PNG grids and stacks private.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN_TAG = "r2"  # the same tag as part 1

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
RUN = PRIVATE / "stage5" / RUN_TAG
assert any((PRIVATE / "export_v1/val").glob("*.npz")), "export_v1/val is missing on Drive: copy it from the laptop folder"
assert (RUN / "record.json").exists(), f"{RUN}/record.json is missing: part 1 must finish first"
assert not (RUN / "eval_record.json").exists(), f"{RUN}/eval_record.json exists: this evaluation is already recorded"
LOCAL_RUN = Path(f"/content/stage5_{RUN_TAG}_eval")  # written first, then copied to RUN on Drive
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)
STARTED = time.perf_counter()

## 2. Models: the step-1000 UNet (strict, hash checked against part 1), frozen VAE and CLIP

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import reference_ddim_scheduler
from rhuh_synthesis.overfit import load_trained_unet
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
S5, EVALUATION = config["stage5_pilot"], config["evaluation"]
VALIDATION, DIAGNOSTICS = S5["validation"], S5["diagnostics"]
CODEC = config["representation"]["mask_codec"]
part1 = json.loads((RUN / "record.json").read_text(encoding="utf-8"))
WEIGHTS = RUN / part1["weights"]["file"]
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, hf_hub_download(MSF_REPO, MSF_FILE))
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
unet11, weights_report = load_trained_unet(unet4, WEIGHTS, device="cuda")
del unet4
torch.cuda.empty_cache()
assert weights_report["sha256"] == part1["weights"]["sha256"], "The weights on Drive differ from part 1: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
ddim = reference_ddim_scheduler()
record = {
    "stage": "5_pilot_eval", "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "config_sha256": config_sha256(config), "part1_config_sha256": part1["config_sha256"], "stage5_config": S5,
    "part1_commit": part1["commit"],
    "environment": environment(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0), "weights": weights_report,
}
print({k: record[k] for k in ("commit", "config_sha256", "torch", "cuda", "gpu")}, weights_report)
changed = [name for name, block in part1["frozen_blocks"].items() if config[name] != block]
assert not changed, f"Frozen config blocks changed after part 1: {changed}. Stop and report"


## 3. Validation patients, copy controls and pre-op conditions; training thumbnails for the copying check

In [ ]:
from rhuh_synthesis.controls import copy_controls
from rhuh_synthesis.pilot import accepted_patients, condition_cache, intensity_matched_copy, local_copies, mask_extras, thumbnails
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.stage3 import clip_encoder
from rhuh_synthesis.working_grid import to_working_grid

split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
assigned = split["roles"]["val"]["patients"]
val_dir = PRIVATE / "export_v1/val"
patients = accepted_patients(assigned, val_dir)
assert not set(patients) & set(split["roles"]["test"]["patients"]), "A test patient reached this run: stop and report"
letters = {p: f"V{n + 1}" for n, p in enumerate(patients)}
FACTOR = DIAGNOSTICS["copying"]["thumbnail_factor"]
started = time.perf_counter()
train_patients = [p for p in split["roles"]["train"]["patients"] if p not in policy["excluded_training"]]
paths = {p: val_dir / f"{p}.npz" for p in patients}
paths.update({p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in train_patients})
paths = local_copies(paths, "/content/stage5_exports")  # from here on, reading the data does not need Drive
arrays, crops, grids, copies = {}, {}, {}, {}
for patient in patients:
    with np.load(paths[patient]) as data:
        arrays[patient] = {key: data[key] for key in data.files}
    crops[patient], grids[patient] = to_working_grid(arrays[patient])
    grids[patient]["post_support"] = crop_forward(arrays[patient]["post_support"].astype(np.uint8), crops[patient]).astype(bool)
    copies[patient] = copy_controls(arrays[patient], EVALUATION)
    copies[patient]["intensity_matched_copy"] = intensity_matched_copy(arrays[patient], EVALUATION)
    copies[patient]["pre_label_extras"] = mask_extras(arrays[patient]["pre_label"], arrays[patient], EVALUATION)
cache = condition_cache({p: {k: grids[p][k] for k in ("pre_image", "pre_label", "label")} for p in patients}, vae,
                        clip_encoder(tokenizer, text_encoder, "cuda"), config["text"], mask_codec=CODEC, batch=16, device="cuda")
regions = {p: thumbnails((grids[p]["pre_foreground"] & grids[p]["post_support"]).astype(np.float32), FACTOR) >= 0.5 for p in patients}
target_thumbnails = {p: thumbnails(grids[p]["post_image"], FACTOR) for p in patients}

training_thumbnails, training_id3 = [], []
for patient in train_patients:
    with np.load(paths[patient]) as data:
        grid = to_working_grid({key: data[key] for key in data.files})[1]
    training_thumbnails.append(thumbnails(grid["post_image"], FACTOR))
    id3 = grid["label"] == 3
    training_id3.append(id3[id3.any(axis=(1, 2))])
    del grid
training_thumbnails, training_id3 = np.concatenate(training_thumbnails), np.concatenate(training_id3)
record.update(
    validation={"assigned": len(assigned), "accepted": len(patients), "quarantined_na": len(assigned) - len(patients)},
    letters=letters, training_reference={"patients": len(train_patients), "slices": len(training_thumbnails),
                                         "id3_slices": len(training_id3)},
    setup_seconds=time.perf_counter() - started,
)
print(record["validation"], record["training_reference"], f"| setup took {record['setup_seconds']:.0f}s")
for patient in patients:
    copy = copies[patient]["mri_copy"]
    print(letters[patient], "copy primary MAE", round(copy["primary_mae"], 4), "| stable", round(copy["stable_mae"], 4),
          "| intensity-matched copy primary", round(copies[patient]["intensity_matched_copy"]["primary_mae"], 4),
          "| reference ID 3 mL", round(copies[patient]["pre_label_copy"]["reference_id3_ml"], 3),
          "| slices", len(cache["patients"][patient]["prompts"]))

## 4. Generate every stack (about 1.5 to 2 hours)

Finished stacks are skipped on a rerun, including stacks from an earlier session that are on Drive. If the GPU runs out of memory the batch is halved for that stack and the later ones; the batch used is recorded per stack.

In [ ]:
from rhuh_synthesis.pilot import generate_stack, halving_batch, mirror, release_maps_during_forward, save_stack

GENERATE = dict(num_inference_steps=config["sampling"]["num_inference_steps"],
                guidance_scale=config["sampling"]["guidance_scale"], mask_codec=CODEC)
RUNS = [(p, sampler, seed) for p in patients for sampler, spec in VALIDATION["samplers"].items() for seed in spec["seeds"]]
STACKS = LOCAL_RUN / "stacks"
TIMINGS = LOCAL_RUN / "stack_timings"
TIMINGS.mkdir(parents=True, exist_ok=True)
for folder in ("stacks", "stack_timings"):
    print(f"{folder} from Drive:", mirror(RUN / folder, LOCAL_RUN / folder))


def to_drive():
    report = mirror(LOCAL_RUN, RUN)
    if report["error"] is not None:
        print("copy to Drive failed:", report, "(generation continues, the copy is retried later)")
    return report



def stack_name(patient, sampler, seed):
    return f"{letters[patient]}_{sampler}_seed{seed}"


batch = VALIDATION["batch_pairs"]
torch.cuda.reset_peak_memory_stats()
with release_maps_during_forward(unet11):
    for patient, sampler, seed in RUNS:
        name = stack_name(patient, sampler, seed)
        path = STACKS / f"{name}.npz"
        if path.exists():
            print("kept", name)
            continue
        start = time.perf_counter()
        (images, masks), batch = halving_batch(
            lambda size: generate_stack(unet11, vae, ddim, cache, patient, sampler=sampler, seed=seed, batch=size, **GENERATE),
            batch)
        save_stack(path, images, masks)
        seconds = time.perf_counter() - start
        (TIMINGS / f"{name}.json").write_text(json.dumps({"seconds": seconds, "batch": batch, "slices": len(images)}), encoding="utf-8")
        to_drive()
        print(f"{name}: {seconds:.0f}s, batch {batch}, {len(images) / seconds:.2f} pairs/s, peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
record["stacks"] = {path.stem: json.loads(path.read_text(encoding="utf-8")) for path in sorted(TIMINGS.glob("*.json"))}
record["generation_peak_gib_this_session"] = torch.cuda.max_memory_allocated() / 2**30
missing = [stack_name(*run) for run in RUNS if not (STACKS / f"{stack_name(*run)}.npz").exists()]
assert not missing, f"Missing stacks: {missing}"
print(f"{len(RUNS)} stacks ready")

## 5. Score from the saved stacks in the 1 mm frame

In [ ]:
from rhuh_synthesis.pilot import load_stack, nearest_training, nearest_training_masks, score_stack, seed_diversity

started = time.perf_counter()
TOP_K = DIAGNOSTICS["copying"]["top_k"]
copying_reference = {letters[p]: nearest_training(thumbnails(grids[p]["pre_image"], FACTOR), regions[p], training_thumbnails,
                                                  target_thumbnails[p], top_k=TOP_K)["closer_to_training_fraction"]
                     for p in patients}
results, diversity, mask_neighbours = {}, {}, {}
for sampler, spec in VALIDATION["samplers"].items():
    results[sampler], diversity[sampler] = {}, {}
    for patient in patients:
        entry = {"copy": copies[patient]["mri_copy"], "matched_copy": copies[patient]["intensity_matched_copy"], "seeds": {}}
        predictions = []
        for seed in spec["seeds"]:
            name = stack_name(patient, sampler, seed)
            images, masks = load_stack(STACKS / f"{name}.npz")
            scores, prediction = score_stack(images, masks, arrays[patient], crops[patient], EVALUATION)
            scores["copying"] = nearest_training(thumbnails(np.nan_to_num(images), FACTOR), regions[patient], training_thumbnails,
                                                 target_thumbnails[patient], top_k=TOP_K)
            mask_neighbours[name] = nearest_training_masks(masks, training_id3, top_k=TOP_K)
            entry["seeds"][seed] = scores
            predictions.append(prediction)
            print(name, "primary", round(scores["image"]["primary_mae"], 4), "stable", round(scores["image"]["stable_mae"], 4),
                  "| ID 3 Dice", scores["mask"]["id3_dice"], "FP lesions", scores["mask"]["false_positive_lesions"],
                  "| closer to training", round(scores["copying"]["closer_to_training_fraction"], 3))
        results[sampler][letters[patient]] = entry
        diversity[sampler][letters[patient]] = seed_diversity(predictions, arrays[patient]["pre_foreground"])
        del predictions
record["scoring_seconds"] = time.perf_counter() - started
print(f"scoring took {record['scoring_seconds']:.0f}s")

## 6. Review grids: seed 11 of each sampler, slices chosen by a rule fixed before any output was seen

In [ ]:
from rhuh_synthesis.overfit import prompt_caption, save_overfit_pages
from rhuh_synthesis.pilot import review_slices

GRID_SEED = DIAGNOSTICS["review_grid_seed"]
pages = []
for sampler in VALIDATION["samplers"]:
    for patient in patients:
        images, masks = load_stack(STACKS / f"{stack_name(patient, sampler, GRID_SEED)}.npz")
        grid = grids[patient]
        rows = [{"name": f"{letters[patient]} slice {z}", "caption": prompt_caption(cache, (patient, z)),
                 "pre_image": grid["pre_image"][z], "post_image": grid["post_image"][z], "generated_image": images[z],
                 "label": grid["label"][z], "generated_mask": masks[z]}
                for z in review_slices(grid["pre_label"], grid["pre_foreground"], grid["label"], masks,
                                       evenly=DIAGNOSTICS["review_grid_evenly_spaced"])]
        pages += save_overfit_pages(LOCAL_RUN / "grids" / f"validation_{letters[patient]}_{sampler}.png", rows,
                                    title=f"Stage 5 {RUN_TAG} step {VALIDATION['evaluated_step']} {sampler} seed {GRID_SEED} (validation; class text is an oracle)")
print(len(pages), "pages in", LOCAL_RUN / "grids")

## 7. Reading, write-once record and the part to send

In [ ]:
from rhuh_synthesis.manifest import write_once_json
from rhuh_synthesis.pilot import pilot_reading

reading = pilot_reading(results, S5["reading"], EVALUATION["bootstrap"])
record.update(copies={letters[p]: copies[p] for p in patients}, results=results, reading=reading, diversity=diversity,
              copying_reference=copying_reference, mask_neighbours=mask_neighbours,
              total_seconds_this_session=time.perf_counter() - STARTED)
path, digest = write_once_json(LOCAL_RUN / "eval_record.json", record)
print("saved", path, digest[:12])
for attempt in range(3):
    copied = to_drive()
    if copied["error"] is None:
        break
    drive.mount("/content/drive", force_remount=True)
assert copied["error"] is None, f"Could not copy to Drive. Everything is in {LOCAL_RUN}: keep this session open and report"
print("on Drive:", RUN, copied)


def seed_brief(scores):
    return {"image": {k: scores["image"][k] for k in ("primary_mae", "stable_mae", "primary_ssim", "stable_ssim",
                                                      "primary_excluded_fraction", "stable_excluded_fraction",
                                                      "primary_mae_na_reason", "stable_mae_na_reason")},
            "mask": scores["mask"], "through_plane": scores["through_plane"], "valid": scores["technical"]["valid"],
            "closer_to_training_fraction": scores["copying"]["closer_to_training_fraction"]}


shareable = {
    "commit": commit, "config_sha256": record["config_sha256"][:12], "gpu": record["gpu"],
    "weights_sha256": record["weights"]["sha256"][:12], "validation": record["validation"],
    "training_reference": record["training_reference"], "stacks": record["stacks"],
    "generation_peak_gib_this_session": round(record["generation_peak_gib_this_session"], 1),
    "copies": {letters[p]: {"mri_copy": {k: copies[p]["mri_copy"][k] for k in ("primary_mae", "stable_mae", "primary_ssim", "stable_ssim",
                                                                                "primary_roi_ml", "stable_roi_ml",
                                                                                "primary_excluded_fraction", "stable_excluded_fraction")},
                            "intensity_matched_copy": {k: copies[p]["intensity_matched_copy"][k]
                                                       for k in ("slope", "intercept", "primary_mae", "stable_mae")},
                            "pre_label_copy": copies[p]["pre_label_copy"], "pre_label_extras": copies[p]["pre_label_extras"],
                            "all_background": copies[p]["all_background"]}
               for p in patients},
    "results": {sampler: {letter: {str(seed): seed_brief(scores) for seed, scores in entry["seeds"].items()}
                          for letter, entry in by_letter.items()} for sampler, by_letter in results.items()},
    "reading": reading, "diversity": diversity, "copying_reference": copying_reference,
    "mask_neighbour_best_dice": {name: (round(float(np.mean([row["dice"][0] for row in rows])), 3) if rows else None, len(rows))
                                 for name, rows in mask_neighbours.items()},
}
print(json.dumps(shareable, indent=1))
print("\nVERDICT:", reading["verdict"], "|", reading["checks"])

## Next

Save this notebook with its outputs under a name Git ignores (for example `RHUH_GBM_Colab_stage5_evalipynb.ipynb`), or send Claude the printed JSON block; it has V1 to V3 but no patient IDs. Then download `record.json`, `eval_record.json`, `train_log.jsonl` and the `grids` folder from `synthesis/EXP-2026-001/stage5/<RUN_TAG>/` on Drive into the same folder on your laptop. The stacks and the UNet can stay on Drive.

`replication_justified` means train seeds 2027 and 2028 may be repeated (Stage 6), which still needs your go-ahead. `iterate` means the failed checks are recorded and the next change is declared before it runs. Neither opens the test set.